# Chapter 1 Lab — Transformer Internals: Attention Shapes & KV-Cache (GQA vs MHA)

**Estimated time:** 20 minutes · **Read-and-run — no training, no GPU.**

Trace the attention tensor shapes and the KV-cache memory footprint of a 7B decoder-only
model, and see exactly why Grouped-Query Attention (GQA) shrinks the cache versus
Multi-Head Attention (MHA).

In [ ]:
# A representative 7B decoder-only config (Llama-2-7B-like).
cfg = dict(d_model=4096, n_layers=32, n_heads=32, head_dim=128,
           n_kv_heads_mha=32, n_kv_heads_gqa=8, dtype_bytes=2)  # fp16 = 2 bytes
assert cfg['n_heads']*cfg['head_dim'] == cfg['d_model']
for k,v in cfg.items(): print(f'{k:16s} = {v}')

## 1. Per-token attention shapes
For one token, Q is projected to all query heads; K and V to `n_kv_heads` heads.

In [ ]:
def attn_shapes(n_q_heads, n_kv_heads, head_dim):
    return {'Q': (n_q_heads, head_dim),
            'K': (n_kv_heads, head_dim),
            'V': (n_kv_heads, head_dim)}
print('MHA:', attn_shapes(cfg['n_heads'], cfg['n_kv_heads_mha'], cfg['head_dim']))
print('GQA:', attn_shapes(cfg['n_heads'], cfg['n_kv_heads_gqa'], cfg['head_dim']))

## 2. KV-cache size
Cache holds K **and** V for every layer and every past token:

`bytes = 2 * n_layers * n_kv_heads * head_dim * seq_len * dtype_bytes`

In [ ]:
def kv_cache_bytes(seq_len, n_kv_heads, c=cfg):
    return 2 * c['n_layers'] * n_kv_heads * c['head_dim'] * seq_len * c['dtype_bytes']

for seq_len in (2048, 4096, 32768):
    mha = kv_cache_bytes(seq_len, cfg['n_kv_heads_mha'])
    gqa = kv_cache_bytes(seq_len, cfg['n_kv_heads_gqa'])
    print(f'seq={seq_len:6d}  MHA={mha/1e6:8.1f} MB  GQA={gqa/1e6:8.1f} MB  '
          f'reduction x{mha/gqa:.1f}')

## 3. Your turn
Change `n_kv_heads_gqa` to 1 (that is Multi-Query Attention, MQA) and re-run cell above.
How big is the cache now? Which bottleneck did we just trade away?

## Key takeaway
GQA/MQA attack the **memory/bandwidth** bottleneck of the KV-cache; FlashAttention attacks
the **compute/IO** of the attention kernel; MoE attacks **compute** per token. The exam tests
knowing which efficiency variant targets which bottleneck.